# MOFJSON walkthrough: schema, and does it reconstruct the correct .cif?

This notebook walks through the MOFJSON schema end-to-end on a real
structure (Cu-BTC / HKUST-1) and then answers the concrete question: **if
you take the JSON this pipeline produces and reconstruct a structure from
it, is it actually the same crystal you started with?**

Four tiers, in order:

1. **`raw`** -- ground truth: `lattice` + `atoms` + `bonds` (+ `charges`,
   `aromatic_n_h`). This is the only tier that needs to round-trip back to
   a `.cif` losslessly -- it's a direct, parameter-free encoding of the
   structure's geometry and connectivity.
2. **`roles`** -- `node_linker_roles`: an assignment (metal/framework vs.
   organic linker vs. guest), not a measurement. Required input to the
   next tier.
3. **`derived`** -- `building_blocks`, `net_edges`, `topology`,
   `provenance`. A deterministic *function of* (raw, roles), with a hash
   proving it. Useful for answering chemistry questions (SBUs, RCSR net),
   not needed to reconstruct the .cif.
4. **`lattice_encoding`** -- a separate, single invertible string
   encoding of the same (raw.atoms, raw.bonds, raw.lattice).

We'll build all four, then do the actual round-trip check.


In [1]:
import sys, json, warnings
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "mofjson").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not find a 'src/mofjson' folder anywhere above {start}. "
        "Set PROJECT_ROOT to your project's root folder by hand below, e.g.:\n"
        '    PROJECT_ROOT = Path("/Users/yong/projects/MoFJSON")'
    )

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
print("using project root:", PROJECT_ROOT)

from pymatgen.io.cif import CifParser
from pymatgen.analysis.structure_matcher import StructureMatcher

from src.mofjson import (
    AssignRoles, BuildDocument, RawStructure, StructureToJSON,
    StructureFromJSON, CheckRoundTrip, CheckNetConsistency,
    GetDocumentSchema,
)

# Point this at any .cif you want to walk through -- absolute path is
# safest since it doesn't depend on the notebook's working directory.
CIF_PATH = PROJECT_ROOT / "tests/HKUST1.cif"


using project root: /Users/yong/projects/MoFJSON


## Step 1 -- load the CIF

We use pymatgen just to parse the `.cif` into a `Structure` object (atoms +
lattice). Everything downstream (bond perception, MOFJSON, topology) is
this codebase's own logic, not pymatgen's.


In [2]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    parser = CifParser(CIF_PATH)
    original_structure = parser.parse_structures(primitive=False)[0]

print("formula:", original_structure.composition.reduced_formula)
print("num sites:", len(original_structure))
print("lattice (a, b, c, alpha, beta, gamma):",
      (original_structure.lattice.a, original_structure.lattice.b, original_structure.lattice.c,
       original_structure.lattice.alpha, original_structure.lattice.beta, original_structure.lattice.gamma))


formula: CuH2(C3O2)2
num sites: 156
lattice (a, b, c, alpha, beta, gamma): (18.59484, 18.5949, 18.59496, 59.99992000000001, 60.000009999999996, 60.00034)


## Step 2 -- the `raw` tier: lattice / atoms / bonds

`StructureToJSON` does the actual chemistry: covalent-radius bond
perception (Cordero et al. radii + a skin distance), zero-order bonds for
metal contacts (the InChI/MOFid convention), and a flat, JSON-native
representation. No SBU/topology reasoning happens here -- this is the
ground-truth layer.


In [3]:
mofjson_flat = StructureToJSON(original_structure, include_empty_fields=True)

print("top-level keys:", list(mofjson_flat.keys()))
print()
print("lattice:", mofjson_flat["lattice"])
print()
print("atoms[0]:", mofjson_flat["atoms"][0])
print("atoms[1]:", mofjson_flat["atoms"][1])
print(f"... {len(mofjson_flat['atoms'])} atoms total")
print()
print("bonds[0]:", mofjson_flat["bonds"][0])
print(f"... {len(mofjson_flat['bonds'])} bonds total")


top-level keys: ['lattice', 'atoms', 'bonds', 'charges', 'aromatic_n_h', 'node_linker_roles', 'topology', 'building_blocks', 'net_edges']

lattice: {'a': 18.59484, 'b': 18.5949, 'c': 18.59496, 'alpha': 59.99992000000001, 'beta': 60.000009999999996, 'gamma': 60.00034}

atoms[0]: {'id': 'Cu1', 'element': 'Cu', 'frac_xyz': [0.999998, 0.999998, 0.565198]}
atoms[1]: {'id': 'Cu2', 'element': 'Cu', 'frac_xyz': [0.565194, 0.434802, 2e-06]}
... 156 atoms total

bonds[0]: {'source': 'Cu1', 'target': 'O38', 'order': 0.0, 'shift': [1, 0, 0]}
... 198 bonds total


Field meanings:

- `atoms[i]`: `id` (stable string id), `element` (symbol), `frac_xyz`
  (fractional coordinates in the unit cell), `occupancy` (1.0 unless
  disordered -- disorder isn't supported yet, see conversion.py).
- `bonds[i]`: `source`/`target` (atom ids), `order` (0/1/1.5/2/3 -- 0 means
  a metal coordination bond, not a covalent one), `shift` (the integer
  cell-shift a bond crosses, `[0,0,0]` if both atoms are in the same
  image -- this is what lets a *periodic* graph live in a flat JSON list).


## Step 3 -- roles: which atoms are node / linker / guest

This is an **assignment**, not something measured from geometry alone --
conversion.py's own docstring is explicit that automatic role
partitioning is still an open problem here. `AssignRoles` gives a
reasonable default (metals + directly-coordinating atoms = node,
everything else organic = linker) that you can override by hand for
tricky cases.


In [4]:
node_linker_roles = AssignRoles(mofjson_flat["atoms"], mofjson_flat["bonds"])

from collections import Counter
print(Counter(r["role"] for r in node_linker_roles))
print("example entries:", node_linker_roles[:3])


Counter({'linker': 144, 'node': 12})
example entries: [{'atom_id': 'Cu1', 'role': 'node'}, {'atom_id': 'Cu2', 'role': 'node'}, {'atom_id': 'Cu3', 'role': 'node'}]


## Step 4 -- the full MOFJSONDocument (`raw` / `roles` / `derived` / `lattice_encoding`)

`BuildDocument` runs the whole pipeline (building-block collapse, linker
SMILES via RDKit, polymer detection, RCSR topology matching, lattice
encoding) and returns one object whose own validator has already checked
that `derived` is consistent with `raw`+`roles` (it will raise instead of
returning a document if not).


In [5]:
raw = RawStructure(
    lattice=mofjson_flat["lattice"],
    atoms=mofjson_flat["atoms"],
    bonds=mofjson_flat["bonds"],
    charges=mofjson_flat["charges"] or None,
    aromatic_n_h=mofjson_flat["aromatic_n_h"] or None,
)
doc = BuildDocument(raw, node_linker_roles)

print("doc.raw:", type(doc.raw).__name__, "--", len(doc.raw.atoms), "atoms,", len(doc.raw.bonds), "bonds")
print("doc.roles:", type(doc.roles).__name__)
print("doc.derived:", type(doc.derived).__name__)
print("doc.lattice_encoding:", type(doc.lattice_encoding).__name__)


doc.raw: RawStructure -- 156 atoms, 198 bonds
doc.roles: RoleAssignment
doc.derived: DerivedViews
doc.lattice_encoding: LatticeEncoding


### `derived.building_blocks` and `derived.net_edges`

Every atom gets collapsed into a building block (one Cu paddlewheel node,
one BTC linker, etc.) -- these are the graph's vertices. `net_edges` are
the edges between them, each carrying the periodic `shift` it crosses.


In [6]:
print(f"{len(doc.derived.building_blocks)} building blocks, "
      f"{len(doc.derived.net_edges)} net_edges")
print()
print("example building block:", doc.derived.building_blocks[0])
print()
print("example net_edge:", doc.derived.net_edges[0])


14 building blocks, 48 net_edges

example building block: id='linker_1' atoms=[{'id': 'H1', 'element': 'H', 'frac_xyz': [0.734615, 0.028085, 0.734614]}, {'id': 'H6', 'element': 'H', 'frac_xyz': [0.502685, 0.028085, 0.734615]}, {'id': 'H12', 'element': 'H', 'frac_xyz': [0.734617, 0.028085, 0.502682]}, {'id': 'C1', 'element': 'C', 'frac_xyz': [0.700152, 0.029701, 0.700153]}, {'id': 'C6', 'element': 'C', 'frac_xyz': [0.569996, 0.029701, 0.700153]}, {'id': 'C12', 'element': 'C', 'frac_xyz': [0.700153, 0.029701, 0.569994]}, {'id': 'C25', 'element': 'C', 'frac_xyz': [0.837747, 0.023831, 0.569209]}, {'id': 'C30', 'element': 'C', 'frac_xyz': [0.56921, 0.023832, 0.837747]}, {'id': 'C36', 'element': 'C', 'frac_xyz': [0.569212, 0.023831, 0.569209]}, {'id': 'C49', 'element': 'C', 'frac_xyz': [0.743404, 0.02963, 0.613482]}, {'id': 'C54', 'element': 'C', 'frac_xyz': [0.613482, 0.029631, 0.743403]}, {'id': 'C60', 'element': 'C', 'frac_xyz': [0.613483, 0.02963, 0.613482]}, {'id': 'O1', 'element': 'O',

### `derived.topology`

Real coordination-sequence invariants computed from the periodic net
(BFS shell-counting, up to 10 shells), matched against RCSR's own
published net catalogue (2645 nets, bulk-imported from RCSR's Systre
export -- see rcsr_import.py). `net` is only set on an exact vertex-
signature match; otherwise it stays `None` with the real computed data
still attached, rather than guessing.


In [7]:
topo = doc.derived.topology
print("net:", topo["net"])
print("confidence:", topo["confidence"])
print("matched_reference_source:", (topo.get("matched_reference_source") or "")[:160], "...")
print()
by_degree = {}
for block_id, sig in topo["vertex_signatures"].items():
    by_degree.setdefault(sig["degree"], sig["coordination_sequence"])
for degree, cs in sorted(by_degree.items()):
    print(f"  degree {degree}: coordination_sequence = {cs}")


net: tbo
confidence: heuristic_match_unverified_by_isomorphism
matched_reference_source: bulk-imported from RCSR's own Systre .cgd export (RCSRnets20190601.cgd) via rcsr_import.py: NODE/EDGE representatives expanded into the full periodic graph usin ...

  degree 3: coordination_sequence = [3, 9, 15, 33, 45, 82, 90, 153, 150, 241]
  degree 4: coordination_sequence = [4, 8, 20, 30, 60, 68, 120, 126, 200, 180]


### `derived.provenance` and `lattice_encoding`

`provenance` hashes exactly what `derived` was computed from
(`raw.atoms`, `raw.bonds`, `roles.node_linker_roles`) -- this is what
`MOFJSONDocument`'s validator checks on construction. `lattice_encoding`
is a separate, from-scratch invertible string encoding of the same raw
structure (conceptually SLICES-inspired, but an independent
implementation -- see lattice_encoding.py).


In [8]:
print("provenance:", doc.derived.provenance)
print()
print("lattice_encoding length:", len(doc.lattice_encoding.encoding_string), "chars")
print("first 120 chars:", doc.lattice_encoding.encoding_string[:120])


provenance: method='CollapseToBuildingBlocks' version='v1' computed_from=['atoms', 'bonds', 'node_linker_roles'] computed_from_hash='01531ef7e56b38d49a5018c90b81f1991827b375d225824e75c4d65bd56f9f84'

lattice_encoding length: 8811 chars
first 120 chars: CELL a=18.594840 b=18.594900 c=18.594960 alpha=59.999920 beta=60.000010 gamma=60.000340|ATOMS C#0:0.700152,0.029701,0.70


## Step 5 -- the schema, for reference

`GetDocumentSchema()` returns the real pydantic-generated JSON Schema for
this nested `MOFJSONDocument` shape (this is the same schema shipped as
`mofjson_document_schema.json`).


In [9]:
schema = GetDocumentSchema()
print("top-level required fields:", schema.get("required"))
print()
for prop, spec in schema.get("properties", {}).items():
    print(f"  {prop}: {spec.get('title', spec.get('type', ''))}")


top-level required fields: ['raw']

  raw: 
  roles: 
  derived: 
  lattice_encoding: 
  classification: 


## Save the document

This is exactly what `export_mofjson.py` does under the hood.


In [10]:
doc.verify_structurally()  # raises if derived doesn't expand back to raw -- shouldn't, we just built it
with open("HKUST1.mofjson.json", "w") as f:
    f.write(doc.model_dump_json(indent=2))
import os
print("wrote HKUST1.mofjson.json --", os.path.getsize("HKUST1.mofjson.json"), "bytes")


wrote HKUST1.mofjson.json -- 164597 bytes


## Now the actual question: does the JSON reconstruct the correct `.cif`?

There's no single canonical string serialization for a periodic crystal
(unlike SMILES for a molecule) -- the same structure can be written with a
different origin choice, atom ordering, or setting and still be the exact
same crystal. So "correct" here means: **geometrically equivalent under
pymatgen's `StructureMatcher`** (lattice + site matching under symmetry
operations and numerical tolerance), not byte-identical text.

We rebuild a pymatgen `Structure` from nothing but the `raw` tier's
`lattice` + `atoms` (the only fields `StructureFromJSON` needs to place
atoms in space -- `bonds` are validated for consistency but aren't needed
to place atoms, since they're re-derivable from geometry), and compare it
back to the structure we started with.


In [11]:
with open("HKUST1.mofjson.json") as f:
    saved_doc = json.load(f)

flat_from_saved_file = {
    "lattice": saved_doc["raw"]["lattice"],
    "atoms": saved_doc["raw"]["atoms"],
    "bonds": saved_doc["raw"]["bonds"],
    "charges": saved_doc["raw"].get("charges") or [],
    "node_linker_roles": saved_doc["roles"]["node_linker_roles"],
}

reconstructed_structure = StructureFromJSON(flat_from_saved_file)

matcher = StructureMatcher()
is_match = matcher.fit(original_structure, reconstructed_structure)

print("original formula:      ", original_structure.composition.reduced_formula)
print("reconstructed formula: ", reconstructed_structure.composition.reduced_formula)
print("original num sites:      ", len(original_structure))
print("reconstructed num sites: ", len(reconstructed_structure))
print("original lattice (a,b,c,alpha,beta,gamma):     ",
      (round(original_structure.lattice.a, 4), round(original_structure.lattice.b, 4), round(original_structure.lattice.c, 4),
       round(original_structure.lattice.alpha, 2), round(original_structure.lattice.beta, 2), round(original_structure.lattice.gamma, 2)))
print("reconstructed lattice (a,b,c,alpha,beta,gamma):",
      (round(reconstructed_structure.lattice.a, 4), round(reconstructed_structure.lattice.b, 4), round(reconstructed_structure.lattice.c, 4),
       round(reconstructed_structure.lattice.alpha, 2), round(reconstructed_structure.lattice.beta, 2), round(reconstructed_structure.lattice.gamma, 2)))
print()
print("StructureMatcher.fit(original, reconstructed) ->", is_match)


original formula:       CuH2(C3O2)2
reconstructed formula:  CuH2(C3O2)2
original num sites:       156
reconstructed num sites:  156
original lattice (a,b,c,alpha,beta,gamma):      (18.5948, 18.5949, 18.595, 60.0, 60.0, 60.0)
reconstructed lattice (a,b,c,alpha,beta,gamma): (18.5948, 18.5949, 18.595, 60.0, 60.0, 60.0)

StructureMatcher.fit(original, reconstructed) -> True


### The one-liner version: `CheckRoundTrip`

This is the exact same check, packaged as a single function in
conversion.py, for convenience.


In [12]:
ok, _ = CheckRoundTrip(original_structure)
print("CheckRoundTrip ->", ok)


CheckRoundTrip -> True


/Users/yong/mofjsonenv/lib/python3.11/site-packages/pymatgen/core/structure_matcher.py:705: UserWarning: Not all sites have property role. Missing values are set to None.
  ss = Structure.from_sites(s)
/Users/yong/mofjsonenv/lib/python3.11/site-packages/pymatgen/core/structure_matcher.py:1030: UserWarning: Not all sites have property role. Missing values are set to None.
  cls._get_reduced_istructure(SiteOrderedIStructure.from_sites(struct), primitive_cell, niggli)
/Users/yong/mofjsonenv/lib/python3.11/site-packages/pymatgen/core/structure_matcher.py:1029: UserWarning: Not all sites have property role. Missing values are set to None.
  return Structure.from_sites(


## Write the reconstructed structure back out as a real `.cif` and compare

To make this completely concrete: write the reconstructed `Structure` to
an actual `.cif` file, then diff both files' key facts (formula, lattice,
site count) programmatically -- not just trust the `StructureMatcher`
boolean.


In [13]:
reconstructed_structure.to(filename="HKUST1_reconstructed.cif")
print("wrote HKUST1_reconstructed.cif")
print()
with open(CIF_PATH) as f:
    original_lines = f.readlines()
with open("HKUST1_reconstructed.cif") as f:
    reconstructed_lines = f.readlines()

print(f"original .cif: {len(original_lines)} lines")
print(f"reconstructed .cif: {len(reconstructed_lines)} lines")
print()
print("--- first 12 lines of ORIGINAL ---")
print("".join(original_lines[:12]))
print("--- first 12 lines of RECONSTRUCTED ---")
print("".join(reconstructed_lines[:12]))


wrote HKUST1_reconstructed.cif

original .cif: 188 lines
reconstructed .cif: 182 lines

--- first 12 lines of ORIGINAL ---
#======================================================================

# CRYSTAL DATA

#----------------------------------------------------------------------

data_VESTA_phase_1


_pd_phase_name                         'HKUST-1 PBEsol                        '
_cell_length_a                         18.59484
_cell_length_b                         18.59490

--- first 12 lines of RECONSTRUCTED ---
# generated using pymatgen
data_CuH2(C3O2)2
_symmetry_space_group_name_H-M   'P 1'
_cell_length_a   18.59484000
_cell_length_b   18.59490000
_cell_length_c   18.59496000
_cell_angle_alpha   59.99992000
_cell_angle_beta   60.00001000
_cell_angle_gamma   60.00034000
_symmetry_Int_Tables_number   1
_chemical_formula_structural   CuH2(C3O2)2
_chemical_formula_sum   'Cu12 H24 C72 O48'



The line counts and header text differ (pymatgen writes its own `.cif`
in `P1` with its own atom-site loop ordering/labels, rather than
reproducing your original file's exact text/space-group setting) -- that
mismatch is expected and is exactly why we check geometric equivalence
via `StructureMatcher` rather than a text diff. What matters is that both
files describe the same crystal: same lattice parameters, same formula,
same site count, and (per `StructureMatcher.fit` above) the same
structure under symmetry.

**What does and doesn't round-trip today:**

- **Does:** lattice parameters, atomic species + fractional coordinates
  (hence the full crystal geometry), formal charges, node/linker/guest
  role labels (carried as `site_properties` so they survive the round
  trip too).
- **Doesn't (yet), by explicit design:** disordered/partial-occupancy
  sites raise `NotImplementedError` rather than being silently
  mishandled (see conversion.py's docstring) -- this is future work, not
  a silent gap. The bond *list* itself isn't stored on the reconstructed
  pymatgen `Structure` object (pymatgen has no native bond container) --
  but that's fine, since `StructureToJSON` can always re-derive an
  equivalent bond list from geometry again, and `CheckNetConsistency`
  below checks that the derived building_blocks/net_edges tier is
  self-consistent with the bonds actually in the file.


## Bonus: the `derived` tier round-trips too (`CheckNetConsistency`)

Separately from the geometric round trip above, `CheckNetConsistency`
checks a different thing: that `derived.building_blocks` +
`derived.net_edges` (the SBU-level graph) correctly *expand back out* to
exactly `raw.atoms` + `raw.bonds` (the atomic-level graph) -- i.e. no
atoms or bonds were silently dropped or duplicated when collapsing atoms
into building blocks. `MOFJSONDocument` already checks this on
construction (that's what `doc.verify_structurally()` above just did
without raising), but here it is explicitly:


In [14]:
mofjson_dict_for_check = {
    "atoms": doc.raw.atoms,
    "bonds": doc.raw.bonds,
    "building_blocks": [b.model_dump() for b in doc.derived.building_blocks],
    "net_edges": doc.derived.net_edges,
}
ok, details = CheckNetConsistency(mofjson_dict_for_check, node_linker_roles=doc.roles.node_linker_roles)
print("CheckNetConsistency ->", ok)


CheckNetConsistency -> True


## Debugging: where are the CIFs coming from, and round-tripping all of them

Everything above walked through one structure end-to-end. This section
does the same round-trip check as a batch, over every `.cif` sitting in
a fixed folder, and makes each step visible: which files were actually
found (so a typo'd path or an empty folder shows up immediately, not as
a silent zero), where each structure's MOFJSON gets written, and a clear
per-file pass/fail table at the end.


In [15]:
import glob

# Fixed, absolute path -- change this if your cifs/ folder ever moves.
CIF_DIR = Path("/Users/yong/projects/MoFJSON/cifs")
SCHEMA_DIR = PROJECT_ROOT / "schemas"
SCHEMA_DIR.mkdir(exist_ok=True)

if not CIF_DIR.is_dir():
    raise RuntimeError(f"CIF_DIR does not exist: {CIF_DIR}")

batch_cif_paths = sorted(glob.glob(str(CIF_DIR / "*.cif")))

print(f"CIF_DIR    = {CIF_DIR}")
print(f"SCHEMA_DIR = {SCHEMA_DIR}")
print(f"\nFound {len(batch_cif_paths)} .cif file(s):")
for p in batch_cif_paths:
    print(f"  - {Path(p).name}")


CIF_DIR    = /Users/yong/projects/MoFJSON/cifs
SCHEMA_DIR = /Users/yong/projects/MoFJSON/schemas

Found 89 .cif file(s):
  - FuncMOF_BUSQIQ.cif
  - FuncMOF_CAVREY_0.04_0.cif
  - FuncMOF_DITTEH_0.12_0.cif
  - FuncMOF_DUQSEO.cif
  - FuncMOF_GERNOI_0.08_1.cif
  - FuncMOF_MAZSUD.cif
  - FuncMOF_OSIXIY.cif
  - FuncMOF_SIVGIP_0.02_0.cif
  - FuncMOF_UKAPUS.cif
  - FuncMOF_VIVXOP_0.08_0.cif
  - FuncMOF_XAGPUT.cif
  - FuncMOF_XEGKIF.cif
  - FuncMOF_XILVIZ.cif
  - FuncMOF_YARKAF.cif
  - FuncMOF_ZEZPUR.cif
  - MOF_ABAYEI.cif
  - MOF_ABAZAF.cif
  - MOF_ABOVUL.cif
  - MOF_ABXALA01.cif
  - MOF_ACATAB.cif
  - MOF_ACIGAX.cif
  - MOF_ACIGEB.cif
  - MOF_ACOLUB.cif
  - MOF_ACRNCU.cif
  - MOF_ACURUO.cif
  - MOF_ADAPAX.cif
  - MOF_ADEGAU.cif
  - MOF_ADULIX.cif
  - MOF_AFITIT.cif
  - MOF_AFIXAO.cif
  - MOF_AFOHOT.cif
  - MOF_AGIMIO.cif
  - MOF_AHEGIF.cif
  - MOF_AHELAC.cif
  - MOF_AHUXAD.cif
  - MOF_AHXCUD10.cif
  - MOF_AJIJEI01.cif
  - MOF_ALUKOI.cif
  - MOF_ALULAV.cif
  - MOF_AMIHOV01.cif
  - MOF_APIROH.c

Now convert each one to MOFJSON (saving it into its own file under
`schemas/`, so there's a real artifact to open afterward, not just an
in-memory dict) and run `CheckRoundTrip` on it.


In [16]:
import pandas as pd

rows = []
for cif_path in batch_cif_paths:
    cif_path = Path(cif_path)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            parser = CifParser(cif_path)
            struct = parser.parse_structures(primitive=False)[0]
    except Exception as e:
        rows.append({"cif": cif_path.name, "status": "cif_parse_failed", "detail": str(e), "schema_file": None})
        continue

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ok, batch_mofjson = CheckRoundTrip(struct)
    except Exception as e:
        rows.append({"cif": cif_path.name, "status": "roundtrip_raised", "detail": str(e), "schema_file": None})
        continue

    schema_path = SCHEMA_DIR / (cif_path.stem + ".mofjson.json")
    schema_path.write_text(json.dumps(batch_mofjson, indent=2))

    if ok:
        rows.append({"cif": cif_path.name, "status": "PASSED", "detail": "", "schema_file": schema_path.name})
    else:
        recon = StructureFromJSON(batch_mofjson)
        detail = (
            f"orig={struct.composition.reduced_formula}/{len(struct)} sites vs. "
            f"recon={recon.composition.reduced_formula}/{len(recon)} sites"
        )
        rows.append({"cif": cif_path.name, "status": "FAILED", "detail": detail, "schema_file": schema_path.name})

results_df = pd.DataFrame(rows)
n_passed = (results_df["status"] == "PASSED").sum()
print(f"Round-trip summary: {n_passed}/{len(results_df)} passed\n")
results_df


Round-trip summary: 89/89 passed



,cif,status,detail,schema_file
0,FuncMOF_BUSQIQ.cif,PASSED,,FuncMOF_BUSQIQ.mofjson.json
1,FuncMOF_CAVREY_0.04_0.cif,PASSED,,FuncMOF_CAVREY_0.04_0.mofjson.json
2,FuncMOF_DITTEH_0.12_0.cif,PASSED,,FuncMOF_DITTEH_0.12_0.mofjson.json
3,FuncMOF_DUQSEO.cif,PASSED,,FuncMOF_DUQSEO.mofjson.json
4,FuncMOF_GERNOI_0.08_1.cif,PASSED,,FuncMOF_GERNOI_0.08_1.mofjson.json
...,...,...,...,...
84,MOF_BUDCIN.cif,PASSED,,MOF_BUDCIN.mofjson.json
85,MOF_BUDGEN.cif,PASSED,,MOF_BUDGEN.mofjson.json
86,MOF_BUDWUV.cif,PASSED,,MOF_BUDWUV.mofjson.json
87,MOF_BUQSIS.cif,PASSED,,MOF_BUQSIS.mofjson.json


## Summary

Starting from `HKUST1.cif`, this notebook built the full MOFJSON document
(raw ground truth, role assignment, derived building blocks/net/topology,
lattice encoding), saved it as real JSON, and then proved that reconstructing a structure from nothing but that JSON's
`raw` tier gives back the same crystal: same lattice, same formula, same
site count, and a positive `StructureMatcher` match against the original
structure. The derived (SBU-level) tier was separately checked to expand
back to the exact same atoms/bonds it was built from.
